# Layer 2 — safe session routing with the Microsoft Agent Framework

One agent (one service principal) serves many users against the confidential MCP
sandbox. The rule that stops users' data from being mixed up:

> **The model never chooses `session_id` or `user_id`.** Trusted code sets them from
> the authenticated user. The model only sees a business tool: “compute the averages.”

It pairs with **Layer 1**: each call carries `X-User-Id`, so the enclave owns every
session as `(agent + user)` and denies cross-user access even if the agent slips.

## 1. Install dependencies

On Microsoft-managed devices public PyPI is blocked; install via the proxy (drop
`--index-url` on an open network).

In [ ]:
%pip install --index-url https://packagefeedproxy.microsoft.io/pypi/simple \
    "agent-framework>=1.0.0b1" "azure-identity>=1.17" "mcp>=1.9,<2" \
    "pandas>=2.0" "openpyxl>=3.1" "httpx>=0.27" "python-dotenv>=1.0"

## 2. Configuration (from `.env`)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

TENANT_ID     = os.environ["MCP_TENANT_ID"]
CLIENT_ID     = os.environ["MCP_CLIENT_ID"]        # the agent's service principal
CLIENT_SECRET = os.environ["MCP_CLIENT_SECRET"]
API_APP_ID    = os.environ["MCP_API_APP_ID"]
SCOPE         = f"api://{API_APP_ID}/.default"
MCP_ENDPOINT  = os.environ["MCP_ENDPOINT"]

AOAI_ENDPOINT   = os.environ["AZURE_OPENAI_ENDPOINT"]
AOAI_DEPLOYMENT = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"]

## 3. Call the sandbox as the agent, stamped with the user's id

`sandbox()` makes one MCP call. It authenticates as the agent's service principal and
adds `X-User-Id` (Layer 1). The `sessions` dict maps each user to their own session —
it lives here in trusted code, never exposed to the model.

In [ ]:
import base64, httpx
from pathlib import Path
from azure.identity import ClientSecretCredential
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

credential = ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
sessions = {}  # authenticated user -> session id


def _client(headers=None, timeout=None, auth=None):
    # Trust the enclave's self-signed cert; authenticity comes from attestation.
    return httpx.AsyncClient(headers=headers, timeout=timeout, auth=auth, verify=False)


async def sandbox(user_id, tool, args):
    headers = {
        "Authorization": f"Bearer {credential.get_token(SCOPE).token}",
        "X-User-Id": user_id,
    }
    async with streamablehttp_client(MCP_ENDPOINT, headers=headers,
                                     httpx_client_factory=_client) as (r, w, _):
        async with ClientSession(r, w) as mcp:
            await mcp.initialize()
            result = await mcp.call_tool(tool, args)
            return result.content[0].text


async def get_session(user_id):
    if user_id not in sessions:
        sessions[user_id] = await sandbox(user_id, "create_session", {})
    return sessions[user_id]

## 4. Build an agent for one authenticated user

The glue uploads the user's file first, then defines the tool. `compute_column_averages`
closes over `user_id` and takes **no arguments** — the model cannot name a user or
session, so it cannot reach anyone else's data.

In [ ]:
import textwrap
from agent_framework import ai_function, ChatAgent
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import DefaultAzureCredential

chat_client = AzureOpenAIChatClient(endpoint=AOAI_ENDPOINT,
                                    deployment_name=AOAI_DEPLOYMENT,
                                    credential=DefaultAzureCredential())


async def make_agent(user_id, xlsx_path):
    session_id = await get_session(user_id)
    data_b64 = base64.b64encode(Path(xlsx_path).read_bytes()).decode()
    await sandbox(user_id, "upload_file",
                  {"session_id": session_id, "path": "data.xlsx", "content_base64": data_b64})

    @ai_function(description="Average every numeric column in the user's spreadsheet.")
    async def compute_column_averages() -> str:
        code = textwrap.dedent('''
            import pandas as pd
            df = pd.read_excel("data.xlsx")
            print(df.select_dtypes("number").mean().to_string())
        ''')
        return await sandbox(user_id, "run_python",
                             {"session_id": sessions[user_id], "code": code})

    return ChatAgent(chat_client=chat_client,
                     instructions="Analyze the user's spreadsheet with the tool. "
                                  "You cannot access anyone else's data.",
                     tools=[compute_column_averages])

## 5. Demo data for two users

In [ ]:
import pandas as pd

pd.DataFrame({"region": ["East", "West"], "revenue": [120.0, 200.0], "units": [12, 20]}
            ).to_excel("alice_sales.xlsx", index=False)   # revenue 160, units 16
pd.DataFrame({"region": ["North", "South"], "revenue": [300.0, 500.0], "units": [30, 50]}
            ).to_excel("bob_sales.xlsx", index=False)     # revenue 400, units 40

## 6. Run both users through the same agent process

Each gets an agent bound to their identity; each answer reflects only their own data.

In [ ]:
alice = await make_agent("alice@contoso.com", "alice_sales.xlsx")
bob   = await make_agent("bob@contoso.com", "bob_sales.xlsx")

print("ALICE:", (await alice.run("What is the average revenue and units?")).text)
print("BOB:  ", (await bob.run("What is the average revenue and units?")).text)
print("sessions:", sessions)   # two distinct ids, held only by trusted code

## Why a mix-up can't happen

- The tool takes no identity arguments, so the model can't address another user.
- Routing lives in `sandbox()` / `sessions`, never shown to the model.
- The server backs it up: `X-User-Id` makes the enclave own each session as
  `(agent + user)` and deny cross-user access (Layer 1).
- All of it runs inside the attested, memory-encrypted TEE — invisible to admins.